# ROAD-SHIELD: real-data detector training
Trains one YOLO detector for potholes (incl. water-filled) and cracks on human-labelled data only, with duplicate-clustered splits, a selection rule fixed before test scoring, and measured-only metrics. **Runtime: GPU (T4)** - Runtime > Change runtime type.

Run the cells in order: 1 setup, 2 Kaggle data, 3 optional samples, 4 licences, 5 train, 6 metrics.

In [ ]:
!pip -q install ultralytics opencv-python-headless pyyaml kagglehub
# the training scripts live on this branch (master does not carry them)
!git clone -q --depth 1 -b detector-only-real-training https://github.com/udbhav968-creator/SIH_PROJECT.git /content/repo
%cd /content/repo
!ls training

## 2. Real labelled data from Kaggle
Upload `kaggle.json` (Kaggle > Settings > API > Create token) to the Colab Files panel first.
Pick datasets that have **bounding-box labels made by people** (not classification-only sets). Open each dataset's Kaggle page and note its licence. Replace the examples below with the datasets you chose - they are examples to verify, not a recommendation. If the script stops on an unknown class name, add `--map "raw_name=pothole"`.

In [ ]:
!mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# one --dataset per source: "<owner>/<slug>:<target classes inside it>"
!python training/prepare_kaggle_detection_data.py --out /content/data \
    --dataset andrewmvd/pothole-detection:pothole \
    --dataset chitholian/annotated-potholes-dataset:pothole

In [ ]:
# OPTIONAL: add the 5 hand-annotated sample photos (30 pothole boxes) as extra training examples
!python training/export_samples_as_source.py --data-root /content/data

In [ ]:
# Licences: print each source, then fill in the real licence from its Kaggle page.
import json
p = '/content/data/sources.json'
s = json.load(open(p))
for k, v in s.items():
    print(k, '|', v['url'], '|', v['licence'], '|', v.get('n_images'))

LICENCES = {
    # "<source name printed above>": "<licence text from its page>",
}
for k, lic in LICENCES.items():
    s[k]['licence'] = lic
json.dump(s, open(p, 'w'), indent=2)

In [ ]:
!python training/train_detectors_real.py --data-root /content/data --out /content/out --epochs 100

In [ ]:
import json; print(json.dumps(json.load(open('/content/out/detector_metrics.json')), indent=2)[:4000])

## Send the result back
Download `/content/out/detector_metrics.json` and `/content/out/train/weights/best.onnx` (Files panel > right-click > Download) and send them. Do not edit them. The pipeline draws no boxes from a detector until `scripts/verify_rdd_detector.py` has checked it.